*Mount Google Drive. Change if using a different location*

In [ ]:
# Change file path depending on the location of the data

from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/warehouse-bottleneck-analyzer"
MODELS_PATH = f"{BASE_PATH}/models"
RAW_DATA_PATH = f"{BASE_PATH}/data/raw/warehouse_line_throughput_synthetic.csv"
NOTEBOOKS_PATH = f"{BASE_PATH}/notebooks"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


*Data download and preprocessing*

In [ ]:
import pandas as pd
import numpy as np

# Read the file, ensure "timestamp" is in date-time format
# then set it as the index.

df = pd.read_csv(RAW_DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
df = df.set_index("timestamp")

*This code verifies if planned_work is 0 in the synthetic dataset. The model was designed to predict next-hour backlog risk during periods with planned incoming work. Scenarios with zero planned work and an existing backlog fell outside the model's training domain and were not assigned a model-derived probability.*

In [ ]:
edge_case = df[
    (df["planned_work"] == 0)
]

print(edge_case.shape)
print(edge_case.head())

(0, 32)
Empty DataFrame
Columns: [date, hour, line_id, total_stations, shift, weekday, planned_work, stations_down, packers_scheduled, packers_available, avg_units_per_packer, building_extra_packers, other_line_capacity, added_packers, packers_available_after_support, work_diverted_out, scaled_back_work, net_incoming_volume, intervention_type, active_stations, available_work, needed_packers, packers_assigned, reassigned_packers, line_capacity, units_packed, backlog_units, utilization_rate, capacity_shortfall, bottleneck_flag, next_hour_backlog, next_hour_backlog_risk]
Index: []

[0 rows x 32 columns]


*After review of the data through a separate EDA program, **capacity_gap** was the main feature driving operations's behavior. It is recreated here as a function of **backlog_units** and **line_capacity**. Managers care about today's backlog and if the current line capacity can handle it. It is harder or impossible to staff when planned work is very high and capacity is near, or at, maximum. As backlog increases relative to planned work, there is more pressure to drive backlog to zero in the current hour*

In [ ]:
# Engineered features
df["capacity_gap"] = (df["available_work"] - df["line_capacity"])
df["work_pressure"] = (df["backlog_units"] / df["planned_work"])

# Not used in this program but is available if needed.
#df["backlog_planned_ratio"] = (df["backlog_units"] / df["planned_work"])

# Print a few records and verify values of engineered features
print(df[[
    "available_work",
    "line_capacity",
    "capacity_gap",
    "work_pressure"
]].head())


                     available_work  line_capacity  capacity_gap  \
timestamp                                                          
2026-04-01 00:00:00             893            820            73   
2026-04-01 01:00:00             844            840             4   
2026-04-01 02:00:00             732            664            68   
2026-04-01 03:00:00            1013            728           285   
2026-04-01 04:00:00            1004            560           444   

                     work_pressure  
timestamp                           
2026-04-01 00:00:00       0.006719  
2026-04-01 01:00:00       0.000000  
2026-04-01 02:00:00       0.139344  
2026-04-01 03:00:00       0.356751  
2026-04-01 04:00:00       0.622975  


In [ ]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score
)

CLASS_THRESHOLD = 0.39

def predict_at_threshold(estimator, X):
    y_prob = estimator.predict_proba(X)[:, 1]
    return (y_prob >= CLASS_THRESHOLD).astype(int)


def accuracy_at_threshold(estimator, X, y):
    y_pred = predict_at_threshold(estimator, X)
    return accuracy_score(y, y_pred)


def precision_at_threshold(estimator, X, y):
    y_pred = predict_at_threshold(estimator, X)
    return precision_score(y, y_pred, zero_division=0)


def recall_at_threshold(estimator, X, y):
    y_pred = predict_at_threshold(estimator, X)
    return recall_score(y, y_pred, zero_division=0)


def f1_at_threshold(estimator, X, y):
    y_pred = predict_at_threshold(estimator, X)
    return f1_score(y, y_pred, zero_division=0)


def f2_at_threshold(estimator, X, y):
    y_pred = predict_at_threshold(estimator, X)
    return fbeta_score(y, y_pred, beta=2, zero_division=0)

*This helper function returns each model evaluation as a dictionary, with metric and hyperparameter names stored as keys. This provides a lightweight representation of one experiment, and allows the result to be inserted directly as a row in a Pandas DataFrame for side-by-side comparison across models.*

In [ ]:
from sklearn.model_selection import cross_validate

def evaluate_model(
    model,
    model_name,
    parm_changed,
    value,
    X,
    y,
    scoring,
    cv_strategy
):
    """
    Evaluate a Random Forest using cross-validation.

    Returns a dictionary containing:
    - Model identification
    - Hyperparameter values
    - Classification threshold
    - Mean and standard deviation for each CV metric
    """

    cv_results = cross_validate(
        estimator=model,
        X=X,
        y=y,
        scoring=scoring,
        cv=cv_strategy,
        n_jobs=1,
        error_score="raise"
    )

    results = {
        "Model Name": model_name,
        "Parm Changed": parm_changed,
        "Value": value,
        "Threshold": CLASS_THRESHOLD,
        "n_estimators": model.n_estimators,
        "max_depth": model.max_depth,
        "min_samples_leaf": model.min_samples_leaf,
        "min_samples_split": model.min_samples_split
    }

    for metric in scoring:
        scores = cv_results[f"test_{metric}"]

        results[metric.title() + " Mean"] = np.mean(scores)
        results[metric.title() + " Std"] = np.std(scores)

    return results

*There are times where starting with an empty comparison table is needed. This progarm first used it prior to comparing classification metrics for a Random Forest model tuned with different hyperparameter values. It was expanded to compare a Logistic Regression model and a Random Forest model*

In [ ]:
def reset_comparison_table():
    return pd.DataFrame(columns=[
        "Model Name",
        "Parm Changed",
        "Value",
        "Threshold",
        "n_estimators",
        "max_depth",
        "min_samples_leaf",
        "min_samples_split",
        "Accuracy Mean",
        "Accuracy Std",
        "Precision Mean",
        "Precision Std",
        "Recall Mean",
        "Recall Std",
        "F1 Mean",
        "F1 Std",
        "F2 Mean",
        "F2 Std",
        "Roc_Auc Mean",
        "Roc_Auc Std"
    ])

In [ ]:
comparison_table = reset_comparison_table()
classification_threshold = 0.39 # Derived from FC_Operations_Explanatory_Models.ipynb

*Starting with Random Forest as an arbitrary starting point, based only
on explanatory model efforts in an EDA program. It was initially believed there was a non-linear relationship between the target **next_hour_backlog** and the predictive features*

*It was decided the cost of missing backlog risks was higher than false positives. The model contained the feature set determined during EDA. An ablation study was performed to determine which removed features would not materially reduce the final model's ability to predict backlog risk. The study systematically removed one feature at a time and retrained the Random Forest model,the objective being to evaluate the individual contribution of each feature. Features whose removal did not materially reduce the Recall and F2 score metrics were identified as candidates for elimination. This drove model parsimony, resulting in a simpler and more interpretable model without sacrificing predictive accuracy.*

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

y = df["next_hour_backlog_risk"]
X = df[["capacity_gap"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=789,stratify=y)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=20,
    min_samples_split=20,
    random_state=789,
    class_weight="balanced")

rf.fit(X_train, y_train)

y_prob = rf.predict_proba(X_test)[:, 1]
y_pred = predict_at_threshold(rf, X_test)
comparison_table = reset_comparison_table()

*We'll manually tune a standard set of hyperparameters as an exploratory exercise. While known not to be globally optimal, the exercise isolates the influence of each individual hyperparameter. The results of each pass will be captured, and we'll compare their Recall and F2-score scores. We'll keep each set of hyperparameters constant while we'll change one at a time.*
*The following code shows the result of training the baseline model. Each item in the variable **experiment** holds the classification report metric. It is provided to validate the returned dictionary contains the values needed.*

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Create a new result row and add the evaluation
scoring = {
    "accuracy": accuracy_at_threshold,
    "precision": precision_at_threshold,
    "recall": recall_at_threshold,
    "f1": f1_at_threshold,
    "f2": f2_at_threshold,
    "roc_auc": "roc_auc"
}

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=789
)

baseline_experiment = evaluate_model(
    model=rf,
    model_name="Random Forest",
    parm_changed = "Default",
    value = "Baseline",
    X = X_train,
    y = y_train,
    scoring = scoring,
    cv_strategy = cv_strategy
)

for key, value in baseline_experiment.items():
    if isinstance(value, float):
        print(f"{key:12}: {value:.3f}")
    else:
        print(f"{key:12}: {value}")

comparison_table.loc[len(comparison_table)] = baseline_experiment

Model Name  : Random Forest
Parm Changed: Default
Value       : Baseline
Threshold   : 0.390
n_estimators: 100
max_depth   : 5
min_samples_leaf: 20
min_samples_split: 20
Accuracy Mean: 0.710
Accuracy Std: 0.017
Precision Mean: 0.310
Precision Std: 0.016
Recall Mean : 0.899
Recall Std  : 0.049
F1 Mean     : 0.461
F1 Std      : 0.021
F2 Mean     : 0.651
F2 Std      : 0.029
Roc_Auc Mean: 0.882
Roc_Auc Std : 0.022


*The variable **experiment** will now house each prediction result for each changed hyperparameter value. **Experiment** will be loaded into a dataframe for comparison.*

*We start with tuning **n_estimators**.*

In [ ]:
for change_n_estimators in [150, 200, 250, 300, 350, 400, 450, 500]:
    rf = RandomForestClassifier(
        n_estimators=change_n_estimators,
        max_depth=5,
        min_samples_leaf=20,
        min_samples_split=20,
        random_state=789,
        class_weight="balanced"
        )

    rf.fit(X_train, y_train)
    y_prob = rf.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= classification_threshold).astype(int)

    experiment = evaluate_model(
        model=rf,
        model_name="Random Forest",
        parm_changed = "n_estimators",
        value = change_n_estimators,
        X=X_train,
        y=y_train,
        scoring=scoring,
        cv_strategy=cv_strategy
      )

    comparison_table.loc[len(comparison_table)] = experiment

*In this case, all **n_estimator** values of resulted in equal values for Recall. Values of 200, 300, 350, 400, 450, and 500 resulted in the highest values in F2-score, indicating that increasing the number of trees did not improve the model's ability to identify backlog-risk periods. As an additional benefit, Precision values were also highest for the same **n_estimator** values.*

*As the value of 200 provided the same results with lower computational costs, that value will be used in subsequent tuning.*

In [ ]:
# display(comparison_table)
metric_columns = [
    "Accuracy Mean",
    "Precision Mean",
    "Recall Mean",
    "F1 Mean",
    "F2 Mean",
    "Roc_Auc Mean"
]

display(
    comparison_table.style
    .highlight_max(
        subset=metric_columns,
        color="lightgreen"
    )
    .format(precision=4)
)

,Model Name,Parm Changed,Value,Threshold,n_estimators,max_depth,min_samples_leaf,min_samples_split,Accuracy Mean,Accuracy Std,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std,F2 Mean,F2 Std,Roc_Auc Mean,Roc_Auc Std
0,Random Forest,Default,Baseline,0.3900,100,5,20,20,0.7105,0.0173,0.3105,0.0157,0.8990,0.0487,0.4613,0.0211,0.6514,0.0295,0.8816,0.0218
1,Random Forest,n_estimators,150,0.3900,150,5,20,20,0.7053,0.0216,0.3068,0.0179,0.8990,0.0487,0.4571,0.0231,0.6480,0.0301,0.8818,0.0227
2,Random Forest,n_estimators,200,0.3900,200,5,20,20,0.7105,0.0165,0.3104,0.0168,0.8990,0.0487,0.4613,0.0232,0.6515,0.0321,0.8823,0.0223
3,Random Forest,n_estimators,250,0.3900,250,5,20,20,0.7099,0.0172,0.3100,0.0173,0.8990,0.0487,0.4608,0.0238,0.6511,0.0326,0.8829,0.0202
4,Random Forest,n_estimators,300,0.3900,300,5,20,20,0.7105,0.0165,0.3104,0.0168,0.8990,0.0487,0.4613,0.0232,0.6515,0.0321,0.8832,0.0200
5,Random Forest,n_estimators,350,0.3900,350,5,20,20,0.7105,0.0165,0.3104,0.0168,0.8990,0.0487,0.4613,0.0232,0.6515,0.0321,0.8835,0.0199
6,Random Forest,n_estimators,400,0.3900,400,5,20,20,0.7099,0.0176,0.3095,0.0178,0.8949,0.0441,0.4597,0.0243,0.6490,0.0319,0.8834,0.0203
7,Random Forest,n_estimators,450,0.3900,450,5,20,20,0.7093,0.0185,0.3096,0.0183,0.8990,0.0487,0.4604,0.0251,0.6508,0.0337,0.8835,0.0202
8,Random Forest,n_estimators,500,0.3900,500,5,20,20,0.7105,0.0181,0.3099,0.0185,0.8949,0.0441,0.4603,0.0253,0.6494,0.0329,0.8836,0.0203


*Tuning **max_depth** following the same pattern as was done for **n_estimators**.*

In [ ]:
comparison_table = reset_comparison_table()
comparison_table.loc[len(comparison_table)] = baseline_experiment

for change_max_depth in [3, 6, 9, 12, 15, 18]:
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=change_max_depth,
        min_samples_leaf=20,
        min_samples_split=20,
        random_state=789,
        class_weight="balanced",
        n_jobs=-1
        )

    experiment = evaluate_model(
        model=rf,
        model_name="Random Forest",
        parm_changed = "Max Depth",
        value = change_max_depth,
        X=X_train,
        y=y_train,
        scoring=scoring,
        cv_strategy=cv_strategy
      )

    comparison_table.loc[len(comparison_table)] = experiment

*In this case, **max_depth** = 3 and 5 produced defensible results. Smaller max_depth value requires less training time and less memory use, the trees are simpler, has lower overfitting risk and easier interpretations. However, because **min_samples_leaf** = 20 or **min_samples_split** = 20 could be limiting further tree growth, neither value of **max_depth** = 3 or 5 stands out. Both will be kept for now as **min_samples_leaf** and **min_samples_split** are tuned*

In [ ]:
# display(comparison_table)
metric_columns = [
    "Accuracy Mean",
    "Precision Mean",
    "Recall Mean",
    "F1 Mean",
    "F2 Mean",
    "Roc_Auc Mean"
]

display(
    comparison_table.style
    .highlight_max(
        subset=metric_columns,
        color="lightgreen"
    )
    .format(precision=4)
)

,Model Name,Parm Changed,Value,Threshold,n_estimators,max_depth,min_samples_leaf,min_samples_split,Accuracy Mean,Accuracy Std,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std,F2 Mean,F2 Std,Roc_Auc Mean,Roc_Auc Std
0,Random Forest,Default,Baseline,0.3900,100,5,20,20,0.7105,0.0173,0.3105,0.0157,0.8990,0.0487,0.4613,0.0211,0.6514,0.0295,0.8816,0.0218
1,Random Forest,Max Depth,3,0.3900,100,3,20,20,0.6804,0.0272,0.2943,0.0130,0.9365,0.0488,0.4474,0.0138,0.6509,0.0186,0.8887,0.0173
2,Random Forest,Max Depth,6,0.3900,100,6,20,20,0.7186,0.0180,0.3153,0.0190,0.8865,0.0470,0.4650,0.0255,0.6504,0.0333,0.8794,0.0236
3,Random Forest,Max Depth,9,0.3900,100,9,20,20,0.7249,0.0147,0.3193,0.0164,0.8782,0.0518,0.4680,0.0229,0.6500,0.0328,0.8786,0.0236
4,Random Forest,Max Depth,12,0.3900,100,12,20,20,0.7249,0.0147,0.3193,0.0164,0.8782,0.0518,0.4680,0.0229,0.6500,0.0328,0.8785,0.0236
5,Random Forest,Max Depth,15,0.3900,100,15,20,20,0.7249,0.0147,0.3193,0.0164,0.8782,0.0518,0.4680,0.0229,0.6500,0.0328,0.8785,0.0236
6,Random Forest,Max Depth,18,0.3900,100,18,20,20,0.7249,0.0147,0.3193,0.0164,0.8782,0.0518,0.4680,0.0229,0.6500,0.0328,0.8785,0.0236


*Tune **min_samples_leaf**.*

In [ ]:
comparison_table = reset_comparison_table()
comparison_table.loc[len(comparison_table)] = baseline_experiment

for change_max_depth in [3, 5]:
  for change_min_samples_leaf in [1, 5, 9, 13, 17]:
      rf = RandomForestClassifier(
          n_estimators=100,
          max_depth=change_max_depth,
          min_samples_leaf=change_min_samples_leaf,
          min_samples_split=20,
          random_state=789,
          class_weight="balanced",
          n_jobs=-1
          )

      rf.fit(X_train, y_train)
      y_prob = rf.predict_proba(X_test)[:, 1]
      y_pred = (y_prob >= classification_threshold).astype(int)

      experiment = evaluate_model(
          model=rf,
          model_name="Random Forest",
          parm_changed = "Minimum Samples Leaf",
          value = change_min_samples_leaf,
          X=X_train,
          y=y_train,
          scoring=scoring,
          cv_strategy=cv_strategy
          )

      comparison_table.loc[len(comparison_table)] = experiment

*With **max_depth** = 3, the trees are already heavily constrained. Increasing the **min_samples_leaf** may have little additional effect. For **max_depth** = 5, changing **min_samples_leaf** produces more variation in performance, suggesting that the leaf-size restriction has a greater influence when the trees are allowed to grow deeper.*

*This is evidence of a hyperparameter interaction, although there is need to inspect the fitted trees to establish exactly which constraint is binding.*

In [ ]:
# display(comparison_table)
metric_columns = [
    "Accuracy Mean",
    "Precision Mean",
    "Recall Mean",
    "F1 Mean",
    "F2 Mean",
    "Roc_Auc Mean"
]

display(
    comparison_table.style
    .highlight_max(
        subset=metric_columns,
        color="lightgreen"
    )
    .format(precision=4)
)

,Model Name,Parm Changed,Value,Threshold,n_estimators,max_depth,min_samples_leaf,min_samples_split,Accuracy Mean,Accuracy Std,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std,F2 Mean,F2 Std,Roc_Auc Mean,Roc_Auc Std
0,Random Forest,Default,Baseline,0.3900,100,5,20,20,0.7105,0.0173,0.3105,0.0157,0.8990,0.0487,0.4613,0.0211,0.6514,0.0295,0.8816,0.0218
1,Random Forest,Minimum Samples Leaf,1,0.3900,100,3,1,20,0.6867,0.0263,0.2981,0.0148,0.9324,0.0476,0.4512,0.0172,0.6531,0.0223,0.8872,0.0172
2,Random Forest,Minimum Samples Leaf,5,0.3900,100,3,5,20,0.6810,0.0238,0.2940,0.0113,0.9324,0.0476,0.4465,0.0121,0.6491,0.0183,0.8873,0.0175
3,Random Forest,Minimum Samples Leaf,9,0.3900,100,3,9,20,0.6798,0.0250,0.2933,0.0120,0.9324,0.0476,0.4457,0.0128,0.6484,0.0183,0.8877,0.0173
4,Random Forest,Minimum Samples Leaf,13,0.3900,100,3,13,20,0.6815,0.0248,0.2944,0.0115,0.9324,0.0476,0.4470,0.0121,0.6495,0.0178,0.8886,0.0174
5,Random Forest,Minimum Samples Leaf,17,0.3900,100,3,17,20,0.6810,0.0239,0.2950,0.0101,0.9407,0.0529,0.4486,0.0103,0.6533,0.0187,0.8886,0.0174
6,Random Forest,Minimum Samples Leaf,1,0.3900,100,5,1,20,0.7186,0.0253,0.3172,0.0177,0.8948,0.0383,0.4678,0.0189,0.6549,0.0194,0.8797,0.0213
7,Random Forest,Minimum Samples Leaf,5,0.3900,100,5,5,20,0.7145,0.0230,0.3125,0.0170,0.8864,0.0391,0.4618,0.0201,0.6477,0.0234,0.8776,0.0234
8,Random Forest,Minimum Samples Leaf,9,0.3900,100,5,9,20,0.7145,0.0209,0.3129,0.0155,0.8906,0.0450,0.4627,0.0185,0.6497,0.0239,0.8782,0.0237
9,Random Forest,Minimum Samples Leaf,13,0.3900,100,5,13,20,0.7157,0.0205,0.3143,0.0164,0.8949,0.0461,0.4649,0.0202,0.6528,0.0263,0.8794,0.0233


*Tune **min_samples_split**.*

In [ ]:
comparison_table = reset_comparison_table()
comparison_table.loc[len(comparison_table)] = baseline_experiment

for change_max_depth in [3, 5]:
  for change_min_samples_split in [2, 5, 10, 15, 20, 30, 40]:
      rf = RandomForestClassifier(
          n_estimators=100,
          max_depth=change_max_depth,
          min_samples_leaf=20,
          min_samples_split=change_min_samples_split,
          random_state=789,
          class_weight="balanced",
          n_jobs=-1
          )

      rf.fit(X_train, y_train)
      y_prob = rf.predict_proba(X_test)[:, 1]
      y_pred = (y_prob >= classification_threshold).astype(int)

      experiment = evaluate_model(
          model=rf,
          model_name="Random Forest",
          parm_changed = "Minimum Samples Split",
          value = change_min_samples_split,
          X=X_train,
          y=y_train,
          scoring=scoring,
          cv_strategy=cv_strategy
        )

      comparison_table.loc[len(comparison_table)] = experiment

*Seeing that the metrics were identical, model performance was insensitive to **min_samples_split** when tuned independently. That suggests other tree-growth constraints had a greater influence under the baseline settings. Just for this study, a **min_samples_split** of 20 was chosen because it can reduce computational cost.*

In [ ]:
# display(comparison_table)
metric_columns = [
    "Accuracy Mean",
    "Precision Mean",
    "Recall Mean",
    "F1 Mean",
    "F2 Mean",
    "Roc_Auc Mean"
]

display(
    comparison_table.style
    .highlight_max(
        subset=metric_columns,
        color="lightgreen"
    )
    .format(precision=4)
)

,Model Name,Parm Changed,Value,Threshold,n_estimators,max_depth,min_samples_leaf,min_samples_split,Accuracy Mean,Accuracy Std,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std,F2 Mean,F2 Std,Roc_Auc Mean,Roc_Auc Std
0,Random Forest,Default,Baseline,0.3900,100,5,20,20,0.7105,0.0173,0.3105,0.0157,0.8990,0.0487,0.4613,0.0211,0.6514,0.0295,0.8816,0.0218
1,Random Forest,Minimum Samples Split,2,0.3900,100,3,20,2,0.6804,0.0272,0.2943,0.0130,0.9365,0.0488,0.4474,0.0138,0.6509,0.0186,0.8887,0.0173
2,Random Forest,Minimum Samples Split,5,0.3900,100,3,20,5,0.6804,0.0272,0.2943,0.0130,0.9365,0.0488,0.4474,0.0138,0.6509,0.0186,0.8887,0.0173
3,Random Forest,Minimum Samples Split,10,0.3900,100,3,20,10,0.6804,0.0272,0.2943,0.0130,0.9365,0.0488,0.4474,0.0138,0.6509,0.0186,0.8887,0.0173
4,Random Forest,Minimum Samples Split,15,0.3900,100,3,20,15,0.6804,0.0272,0.2943,0.0130,0.9365,0.0488,0.4474,0.0138,0.6509,0.0186,0.8887,0.0173
5,Random Forest,Minimum Samples Split,20,0.3900,100,3,20,20,0.6804,0.0272,0.2943,0.0130,0.9365,0.0488,0.4474,0.0138,0.6509,0.0186,0.8887,0.0173
6,Random Forest,Minimum Samples Split,30,0.3900,100,3,20,30,0.6804,0.0272,0.2943,0.0130,0.9365,0.0488,0.4474,0.0138,0.6509,0.0186,0.8887,0.0173
7,Random Forest,Minimum Samples Split,40,0.3900,100,3,20,40,0.6804,0.0272,0.2943,0.0130,0.9365,0.0488,0.4474,0.0138,0.6509,0.0186,0.8887,0.0173
8,Random Forest,Minimum Samples Split,2,0.3900,100,5,20,2,0.7105,0.0173,0.3105,0.0157,0.8990,0.0487,0.4613,0.0211,0.6514,0.0295,0.8816,0.0218
9,Random Forest,Minimum Samples Split,5,0.3900,100,5,20,5,0.7105,0.0173,0.3105,0.0157,0.8990,0.0487,0.4613,0.0211,0.6514,0.0295,0.8816,0.0218


*Compare tuned parameters against baseline*

In [ ]:
# Uncomment to show just baseline and tuned
comparison_table = reset_comparison_table()
comparison_table.loc[len(comparison_table)] = baseline_experiment

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=20,
    min_samples_split=20,
    random_state=789,
    class_weight="balanced"
)

rf.fit(X_train, y_train)
y_prob = rf.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= classification_threshold).astype(int)

for change_max_depth in [3, 5]:
  if change_max_depth == 3:
    for change_min_samples_split in [2, 50, 100]:
        rf = RandomForestClassifier(
            n_estimators=200,
            max_depth=change_max_depth,
            min_samples_leaf=17,
            min_samples_split=change_min_samples_split,
            random_state=789,
            class_weight="balanced",
            n_jobs=-1
            )

        rf.fit(X_train, y_train)
        y_prob = rf.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= classification_threshold).astype(int)

        experiment = evaluate_model(
            model=rf,
            model_name="Random Forest",
            parm_changed = "Tier 1 parms",
            value = "Tuned",
            X=X_train,
            y=y_train,
            scoring=scoring,
            cv_strategy=cv_strategy
        )

        comparison_table.loc[len(comparison_table)] = experiment

  else:
    for change_min_samples_split in [2, 20, 50]:
        rf = RandomForestClassifier(
            n_estimators=200,
            max_depth=change_max_depth,
            min_samples_leaf=1,
            min_samples_split=change_min_samples_split,
            random_state=789,
            class_weight="balanced",
            n_jobs=-1
            )

        rf.fit(X_train, y_train)
        y_prob = rf.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= classification_threshold).astype(int)

        experiment = evaluate_model(
            model=rf,
            model_name="Random Forest",
            parm_changed = "Tier 1 parms",
            value = "Tuned",
            X=X_train,
            y=y_train,
            scoring=scoring,
            cv_strategy=cv_strategy
        )

        comparison_table.loc[len(comparison_table)] = experiment

# display(comparison_table)
metric_columns = [
    "Accuracy Mean",
    "Precision Mean",
    "Recall Mean",
    "F1 Mean",
    "F2 Mean",
    "Roc_Auc Mean"
]

display(
    comparison_table.style
    .highlight_max(
        subset=metric_columns,
        color="lightgreen"
    )
    .format(precision=4)
)

,Model Name,Parm Changed,Value,Threshold,n_estimators,max_depth,min_samples_leaf,min_samples_split,Accuracy Mean,Accuracy Std,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std,F2 Mean,F2 Std,Roc_Auc Mean,Roc_Auc Std
0,Random Forest,Default,Baseline,0.3900,100,5,20,20,0.7105,0.0173,0.3105,0.0157,0.8990,0.0487,0.4613,0.0211,0.6514,0.0295,0.8816,0.0218
1,Random Forest,Tier 1 parms,Tuned,0.3900,200,3,17,2,0.6833,0.0258,0.2961,0.0143,0.9365,0.0522,0.4495,0.0174,0.6528,0.0248,0.8891,0.0182
2,Random Forest,Tier 1 parms,Tuned,0.3900,200,3,17,50,0.6798,0.0318,0.2943,0.0170,0.9365,0.0488,0.4472,0.0196,0.6508,0.0232,0.8893,0.0180
3,Random Forest,Tier 1 parms,Tuned,0.3900,200,3,17,100,0.6775,0.0318,0.2933,0.0179,0.9408,0.0476,0.4466,0.0205,0.6514,0.0237,0.8894,0.0175
4,Random Forest,Tier 1 parms,Tuned,0.3900,200,5,1,2,0.7192,0.0154,0.3174,0.0122,0.8990,0.0314,0.4690,0.0142,0.6575,0.0174,0.8784,0.0216
5,Random Forest,Tier 1 parms,Tuned,0.3900,200,5,1,20,0.7157,0.0193,0.3149,0.0143,0.8990,0.0314,0.4661,0.0159,0.6552,0.0173,0.8793,0.0217
6,Random Forest,Tier 1 parms,Tuned,0.3900,200,5,1,50,0.7128,0.0213,0.3133,0.0154,0.9032,0.0292,0.4649,0.0170,0.6555,0.0172,0.8804,0.0204


*With cross-validation scoring, we'll answer the questions about how well the model performs on average and its consistency. The results indicate the model behaves quite consistently on both Recall and F2 scores. There is the caveat that the cross-validation effort was only 5-fold, and therefore isn't direct proof of stability.*

In [ ]:
baseline_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=20,
    min_samples_split=20,
    random_state=789,
    class_weight="balanced",
    n_jobs=1
)

baseline_cv = cross_validate(
    baseline_rf,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring=scoring,
    n_jobs=-1
)

# Summarize cross-validation results
baseline_summary = pd.DataFrame({
    "Metric": list(scoring.keys()),
    "Mean": [
        baseline_cv[f"test_{metric}"].mean()
        for metric in scoring
    ],
    "Std": [
        baseline_cv[f"test_{metric}"].std()
        for metric in scoring
    ]
})

display(baseline_summary.round(4))

,Metric,Mean,Std
0,accuracy,0.7105,0.0173
1,precision,0.3105,0.0157
2,recall,0.8990,0.0487
3,f1,0.4613,0.0211
4,f2,0.6514,0.0295
5,roc_auc,0.8816,0.0218


In [ ]:

scores = cross_validate(
    RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=1,
        min_samples_split=2,
        random_state=789,
        class_weight="balanced"),
    X_train,
    y_train,
    cv=cv_strategy,
    scoring=scoring
)
print(
    "Test Recall Mean: ",scores["test_recall"].mean(),
    "\nTest Recall Standard Deviation: ",scores["test_recall"].std(),
    "\nTest F2 Mean: ",scores["test_f2"].mean(),
    "\nTest F2 Standard Deviation: ",scores["test_f2"].std()
    )

Test Recall Mean:  0.8990248226950353 
Test Recall Standard Deviation:  0.03137145679424674 
Test F2 Mean:  0.6575353372253464 
Test F2 Standard Deviation:  0.01735644563070724


*Now that the manual determination of best hyper-parameter values, the next effort will introduce GridSearch. It evaluated every combination of values specified in the parameter grid (the grid was kept small to control the number of fits and reduce the computational cost). For each combination, five-fold cross-validation was performed and the mean F2 score across the validation folds was calculated. The hyperparameter combination producing the highest mean cross-validated F2 score was selected as the optimal model configuration.*

In [ ]:
from sklearn.model_selection import GridSearchCV

# Base model
rf = RandomForestClassifier(
    random_state=789,
    class_weight="balanced",
    n_jobs=-1)

# Parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 20],
    'min_samples_leaf': [1, 17, 20],
    'min_samples_split': [2, 5, 8]
}

# Grid search
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=scoring,
    cv=cv_strategy,
    n_jobs=-1,
    verbose=2,
    refit="f2"
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nFeatures:")
print(X_train.columns.tolist())

print("\nTarget distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

# Fit to the training data only
grid.fit(X_train, y_train)

# Best hyperparameters
print("Best Parameters:", grid.best_params_)
print("Best CV F2:", grid.best_score_)

# Retrieve the best model's cross-validation metrics
best_idx = grid.best_index_

print("\nBest Model CV Results:")
print("Mean Recall:", grid.cv_results_["mean_test_recall"][best_idx])
print("Std Recall:", grid.cv_results_["std_test_recall"][best_idx])
print("Mean F2:", grid.cv_results_["mean_test_f2"][best_idx])
print("Std F2:", grid.cv_results_["std_test_f2"][best_idx])

X_train shape: (1727, 1)
y_train shape: (1727,)

Features:
['capacity_gap']

Target distribution:
next_hour_backlog_risk
0    1489
1     238
Name: count, dtype: int64
next_hour_backlog_risk
0    0.862189
1    0.137811
Name: proportion, dtype: float64
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best Parameters: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best CV F2: 0.6575353372253464

Best Model CV Results:
Mean Recall: 0.8990248226950353
Std Recall: 0.03137145679424674
Mean F2: 0.6575353372253464
Std F2: 0.01735644563070724


*From the previous code block, the best estimator was determined by the hyperparameter combination that achieved the highest mean cross-validated F2 score. Using this model the metrics Recall, Precision, and F1, were computed to provide a more comprehensive evaluation of its predictive performance.*

In [ ]:
from sklearn.metrics import roc_auc_score
best_rf = grid.best_estimator_

y_prob = best_rf.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= CLASS_THRESHOLD).astype(int)

test_results = {
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "f2": fbeta_score(y_test, y_pred, beta=2, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob)
}

for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

precision: 0.3146
recall: 0.9333
f1: 0.4706
f2: 0.6699
roc_auc: 0.8704


*In this code block, the best model was trained, predications calculated, and the metrics were displayed using a classification report.*

In [ ]:
from sklearn.metrics import classification_report

final_report = classification_report(
    y_test,
    y_pred,
    target_names=["No Backlog Risk", "Backlog Risk"],
    digits=4,
    output_dict=True
)

print(final_report)

test_f2 = fbeta_score(
    y_test,
    y_pred,
    beta=2,
    zero_division=0
)

print(f"Final Test F2: {test_f2:.4f}")
print(f"Final ROC AUC: {roc_auc_score(y_test, y_pred):.4f}")

model_metrics_df = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Class": "No Risk",
        "Precision": final_report["No Backlog Risk"]["precision"],
        "Recall": final_report["No Backlog Risk"]["recall"],
        "F1": final_report["No Backlog Risk"]["f1-score"],
        "Support": final_report["No Backlog Risk"]["support"]
    },
    {
        "Model": "Random Forest",
        "Class": "Risk",
        "Precision": final_report["Backlog Risk"]["precision"],
        "Recall": final_report["Backlog Risk"]["recall"],
        "F1": final_report["Backlog Risk"]["f1-score"],
        "Support": final_report["Backlog Risk"]["support"]
    }
])

# Add model-level metrics
model_metrics_df["F2"] = test_f2

model_metrics_df["Accuracy"] = accuracy_score(y_test, y_pred)

model_metrics_df["ROC_AUC"] = roc_auc_score(y_test, y_pred)

{'No Backlog Risk': {'precision': 0.984251968503937, 'recall': 0.6720430107526881, 'f1-score': 0.7987220447284346, 'support': 372.0}, 'Backlog Risk': {'precision': 0.3146067415730337, 'recall': 0.9333333333333333, 'f1-score': 0.47058823529411764, 'support': 60.0}, 'accuracy': 0.7083333333333334, 'macro avg': {'precision': 0.6494293550384853, 'recall': 0.8026881720430108, 'f1-score': 0.6346551400112761, 'support': 432.0}, 'weighted avg': {'precision': 0.891245686985756, 'recall': 0.7083333333333334, 'f1-score': 0.7531479045292239, 'support': 432.0}}
Final Test F2: 0.6699
Final ROC AUC: 0.8027


*Save model classification results to a CSV file*

In [ ]:
import os

os.makedirs(MODELS_PATH, exist_ok=True)

model_metrics_df.to_csv(
    os.path.join(MODELS_PATH, "model_metrics.csv"),
    index=False
)

*Dump files*


In [ ]:
# Uncomment to install
#!pip install joblib

In [ ]:
import joblib, pickle
from datetime import datetime

# Logistic regression model
model_file = os.path.join(MODELS_PATH, "random_forest.pkl")
with open(model_file, "wb") as file:
    pickle.dump(best_rf, file)

print("Model saved:",
    datetime.fromtimestamp(os.path.getmtime(model_file))
)

deployment_metadata = {
    "model_features": [
        "capacity_gap"
    ],
    "positive_class": "Backlog Risk",
    "negative_class": "No Backlog Risk",
    "classification_threshold": classification_threshold,
    "engineered_features": {"capacity_gap": "available_work - line_capacity"}
}

metadata_file = os.path.join(MODELS_PATH, "deployment_metadata.pkl")
with open(metadata_file, "wb") as file:
    pickle.dump(deployment_metadata, file)

print("Metadata saved:",
    datetime.fromtimestamp(os.path.getmtime(metadata_file))
)

Model saved: 2026-10-14 12:41:24
Metadata saved: 2026-10-14 12:41:24
